In [1]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# QEC 101 — Lab 1: The Basics of Classical and Quantum Error Correction — Solutions
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

**What You Will Do:**
* Define the five aspects common to all error correction procedures
* Implement the classical repetition code and analyze its performance
* Construct the generator and parity check matrices for the Hamming code
* Identify the challenges that distinguish quantum error correction from classical methods
* Implement the three-qubit quantum repetition code using CUDA-Q

**Prerequisites:**
* Python and Jupyter familiarity
* Basic knowledge of quantum computing (qubits, gates, measurement, circuit sampling)
* Familiarity with braket notation ($\ket{\psi}$, $\bra{\psi}$)
* Completion of [Quick Start to Quantum Computing with CUDA-Q](https://github.com/NVIDIA/cuda-q-academic/tree/main/quick-start-to-quantum) or equivalent knowledge

**Key Terminology:**
* Encoder
* Decoder
* Logical Codewords
* Codespace
* Error Space
* Noisy Channel
* Logical Error
* Logical Error Rate
* Repetition Code
* Hamming Code
* $[n,k,d]$-codes
* Syndrome
* Parity Checks
* Distance

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes from a kernel
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects simulation or hardware backend
* [`cudaq.NoiseModel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.NoiseModel) — defines a quantum noise model
* [`cudaq.BitFlipChannel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.KrausChannel) — bit-flip noise channel
* [`cudaq.register_operation`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.register_operation) — registers a custom unitary gate

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/qec101/Images ./Images

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt

import cudaq
from cudaq import spin
from cudaq.qis import *

---

## 1.1 The Basics of Error Correction

Classical [Error correction](https://en.wikipedia.org/wiki/Error_correction_code) (EC) is the practice of redundantly encoding data using additional bits such that if some of the data bits are corrupted (flipped from a 0 to a 1 or vice versa) by external noise, the error can be fixed and the integrity of the original data preserved. There are many potential ways to accomplish this and any specific procedure is referred to as an EC code.

It is thanks to EC that you can still watch a DVD that has many scratches, enjoy a clear telephone conversation despite the signal becoming noisy after transmission over long distances, or scan a QR code at a restaurant that is partially obscured. Try it yourself. Pull up a QR code and try covering different parts with your hand and notice that your phone can still interpret the link despite obfuscation of some parts of the grid. EC has its limits, but in many cases can essentially eliminate any negative impacts of noise.

There are five aspects to any general EC procedure: 

1. All EC procedures assume there is some initial information that needs to be preserved or transmitted. The sender and recipient need to interact with the same information even if it is stored in different ways throughout an EC procedure.

2.  An EC procedure first **encodes** the information across $n$ data bits such that $n$ is larger than the minimum number of bits necessary to store the information. For example, the repitition code that we'll cover in the next section uses 3 bits to encode some binary information stored on a single bit by simply repeating the information stored on the single bit three times.

3. The encoding procedure produces **logical codewords** which are the redundant encodings that map to the logical states.  For example, logical 1 could be defined with the codeword 111.

4. A logical codeword then proceeds through a **noisy channel**.  A noisy channel has some probability of randomly corrupting (flipping) any of the data bits. 

5.  The recipient then receives the encoded data and needs to decode it to determine if an error occurred and how they might fix it. Another way to say this, is that the **decoder** takes the message the recipient receives and determines which logical codeword is "closest" to it. If the decoder produces the correct logical codeword, the EC procedure worked. If not, a **logical error** occurred and the recipient incorrectly interprets the message - the worst case scenario.

No EC procedure is perfect, and is usually benchmarked against **[Shannon's limit](https://en.wikipedia.org/wiki/Noisy-channel_coding_theorem)** which is the theoretical limit of the rate of noise free data transfer that can occur through a channel with a given bandwidth and noise level.

In practice, and EC code need to be just "good enough" to ensure that errors do not usually impede the target application. Generally speaking, the more redundancy (extra bits) used in the encoding process, the better the EC will be, but this also comes at the cost of more memory and time to perform more sophisticated encoding and decoding. Thus, there is always a tension between the resources available for error correction and the logical error rate of the procedure necessary for a given application.

---

## 1.2 The Repetition Code

The most basic EC code is called the **repetition code**. 

Consider encoding the information in a single bit (0 or 1). The repetition code simply adds more bits which are in the same state. So a 3-bit repetition code encodes the logical 0 state ($0_L$) as 000 and the logical 1 state ($1_L$) as 111, making 000 and 111 the logical codewords.

Now, assume $0_L$ is transmitted through a noisy channel such that each data qubit has a probability p=0.1 of flipping erroneously. This means there are eight possible states the encoded message could be in after proceeding through the noisy channel. The states can be sorted into two groups. The space of all logical codewords (000 and 111) is called the **codespace**: 

| Codespace (as bitstrings)    | Codespace (as logical states)| 
| ----------- | ----------- | 
| 000 | $0_L$ |
| 111 | $1_L$ |


while the rest of all the potentially received messages belong to the **error space** as shown in the table below.

The job of the decoder is to map encodings in the error space back to a logical codeword in the codespace.  This is usually done through the help of a calculated property that describes the state called a **syndrome**.  In this case, the syndrome is a majority count of the bits. So, the syndrome of the 101 state would be 2 (the count of how many 1's) and corresponds to logical 1 and the syndrome of 001 would be 1.  Additionally, the syndrome of 111 is 3 and the syndrome of 000 os 0.  So we can apply the decoding rule that a message with a syndrome of 2 or 3 would be decoded as 111, and message with a syndrome of 0 or 1 would be decoded as 000.

| Error Space   | Closest logical state to the received message | Error, if only one error occurred in transmission | Syndrome |
| ----------- | ----------- | ----------- | ----------- | 
| 001 | 000 | right most bit | 1| 
| 010 | 000 | middle bit | 1|
| 100 | 000 | left most bit | 1 | 
| 110 | 111 | right most bit | 2| 
| 101 | 111 | middle bit |  2 |
| 011 | 111 |left most bit | 2 |

Any single bit error can be correctly identified and corrected, but two or more bit flips will result in a logical error.  There is no way to know for certain if 110, for example, is a single error from the $1_L$ codeword or two bit errors from the $0_L$ codeword. However, the repetition code still greatly reduces the probability of a logical error compared to no encoding. To be very conservative you could discard results where any error was detected and resend the message.  In this error detection (not correction) case,  a logical error will only occur in the highly unlikely case of three bit flip errors, where 000 was transmitted but 111 was received.  This approach requires additional data transfers to compensate for the cases where errors occurred. Error correction can eliminate the need for extra data transfers at the expense of possibly mistaking a 2-bit error for a 1-bit error.  This hints at an important property of error correction codes: distance.

Codes are characterized with the so called [n,k,d] notation where n is the number of data bits used to encode k logical bits and d is the **distance**. The distance is the number of errors that must occur for one logical codeword to become the next closest codeword. In our example of the three bit repetition code $n=3$, $k = 1$, and $d=3$, so we refer to this as a [3,1,3] code. The number of errors a code can correct $t$ is a function of code distance and can be calculated as $t = \text{floor}[(d-1)/2]$

The table below shows the likelihood of the four possible scenarios below. Notice the three bit repetition code with majority count will transmit the message with 0.972 probability of success, a significant improvement over the original probability of 0.9.  The **logical error rate** is equal to $1-p$, where $p$ is the probability of success.  In the case of the 3-bit repetition code, the logical error rate is 0.028.

<img src="../Images/errortable.png" alt="Table showing the four possible outcomes of the three-bit repetition code with their associated probabilities: no errors, one correctable error, two errors causing logical failure, and three errors causing logical failure" style="width: 600px;"/>



<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1:</span>** The Repetition Code

You now know enough to code up the repetition code. The exercise below will require you to generalize the repetition code so it will work with $n$ bits. Fill in the `##TODO##` sections and then observe the plots that are generated. What conclusions can you draw from the code performance using more bits? What do you notice about the logical error rate relative to the physical error rate?

</div>

In [ ]:
# EXERCISE 1

def encode(bit, n):
    """Function that encodes a single bit rendundantly n times

    Parameters
    ----------
    bit: int
        Input bit (1 or 0)
    n : int
        repetitions to use for encoding

    Returns
    -------
    str
        string of length n redundantly encoding bit
    """
    return [bit] * n

def decode(bits):
    """Function that decodes a message using majority voting to determine the closest codeword

    Parameters
    ----------
    bits: str
        bitstring corresponding to message that has passed through noisy channel

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    return 1 if sum(bits) > len(bits) // 2 else 0

def transmit(bits, p_error):
    """Function that receives a codeword, and randomly flips each bit with probability p_error to emulate transmission through noisy channel
    
    Parameters
    ----------
    bits: str
        bitstring corresponding to an encoded message without noise
    p_error: float
        probability that a bit will flip through transmission

    Returns
    -------
    int
        1 or 0 corresponding to decoded codeword
    """
    return [1 - bit if random.random() < p_error else bit for bit in bits]

def simulate_logical_error_rate(n, p_error, trials):    
    """Function to determine the logical error rate of an n-bit repetition code over specified number of trials.
    
    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate

    Returns
    -------
    float
        The logical error rate `n_errors/trials`
    """   
    errors = 0
    for _ in range(trials):
        bit = random.randint(0, 1)
        encoded = encode(bit, n)
        received = transmit(encoded, p_error)
        decoded = decode(received)
        if decoded != bit:
            errors += 1
    return errors / trials

def plot_logical_vs_physical_error_rate(n, trials):
    """Function to plot logical vs physical error rate for fixed n and number of trials.
    
    Parameters
    ----------
    n: int
        specifies n-bit repetition code to use
    trials: int
        number of trials used to determine logical error rate
    """  

    p_values = [i * 0.01 for i in range(51)]  # Physical error rates from 0 to 0.3
    logical_error_rates = [simulate_logical_error_rate(n, p, trials) for p in p_values]

    plt.figure(figsize=(10, 6))
    plt.plot(p_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs Physical Error Rate')
    plt.xlabel('Physical Error Rate')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

def plot_logical_vs_repetitions(p_error, max_n, trials):
    """Function to plot logical error rate vs bits used for redundant encoding
    
    Parameters
    ----------
    max_n: int
        specifies the maximum n-bit repetition code to use
    p_error: float
        probability that a bit will flip through transmission
    trials: int
        number of trials used to determine logical error rate
    """   
    n_values = range(1, max_n + 1)  # Repetition sizes from 1 to max_n
    logical_error_rates = [simulate_logical_error_rate(n, p_error, trials) for n in n_values]

    plt.figure(figsize=(10, 6))
    plt.plot(n_values, logical_error_rates, marker='o')
    plt.title('Logical Error Rate vs Number of Repetitions (n)')
    plt.xlabel('Number of Repetitions (n)')
    plt.ylabel('Logical Error Rate')
    plt.grid(True)
    plt.show()

# Example Usage
n = 3         # Number of repetitions for the first plot
p_error = 0.1       # Physical error rate for the second plot
trials = 10000

# Generate the plots
plot_logical_vs_physical_error_rate(n, trials)
"""Notice how the logical error rate is an improvement over the physical error rate as long as an error is less that 50% likley."""

max_n=20
plot_logical_vs_repetitions(p_error, max_n, trials)
"""Encoding the data with more bits improves the logical error rate to the point where logical errors are virtually nonexistent. 
Notice how going from n to n+1 when n is odd does not add much improvement as even numbers can result in voting where a tie occurs."""

---

## 1.3 More Efficient EC Codes (The Hamming Code)

There are many clever ways to improve the efficiency of EC codes. One common way is to make use of a concept called **parity checks**. Parity checks provide a clever way to index where errors occur, without a brute force statistical approach like the repetition code. 

[The Hamming code](https://en.wikipedia.org/wiki/Hamming_code) is a great example of a parity check code.  The [7,4,3] Hamming code consists of four data bits ($d_1, d_2, d_3, d_4$) and three additional parity check bits ($p_1, p_2, p_3$). This example will only consider single bit errors as this is another distance 3 code and can only correct single bit errors. 

This is accomplished by each parity bit encoding a parity, or the mod2 sum of a subset of the data bits.  The [Venn diagram](https://en.wikipedia.org/wiki/Hamming_code) below depicts the encoding.  In this example, $p_1$ encodes the parity of $d_1$, $d_2$, and $d_4$.  If our data bits ($d_1d_2d_3d_4$) were 0110, then $p_1$ would be calculated to be 1.

<img src="../Images/Hamming(7,4).svg" alt="Venn diagram illustrating the [7,4,3] Hamming code structure with three overlapping circles representing parity check bits p1, p2, and p3, and four data bits d1 through d4 placed in the intersecting regions" style="width: 300px;"/>

Using the Hamming code widget [here](https://nvidia.github.io/cuda-q-academic/interactive_widgets/hamming.html), reason through the following example:

> If you wanted to send the message 0110 (here $d_1 = 0$, $d_2 = 1$, $d_3 = 1$, and $d_4 = 0$), appending the three parity bits to the end of the original bitstring would produce the logical codeword: 0110110 (where $p_1 = 1$, $p_2 = 1$, and $p_3 = 0$).  Note, this is a slight deviation from the traditional placement of the bits in the Hamming code done for simplicity.
>
>Errors could occur on any of the data or parity bits. Assume an error occurs on $d_2$ and the recipient receives 0010110. To produce the syndrome, the recipient can take the received data bits, 0010, and compute the expected parity. This is then compared to the parity that was sent, 110. The parity bits that disagree flag an error. 
>
>In this case, the received message has parity bits 011 which disagrees with 110. Here, $p_1$ and $p_3$ are flagged.  This syndrome can only correspond to an error on $d_2$ based on the Venn diagram.  




The Hamming code takes advantage of the fact that the parity bits can encode up to $2^3 = 8$ syndromes, more than enough to consider the seven possible single bit flip errors that could occur. This means, 4 bits can be encoded with 7 bits which is an improvement over the $n$ to 1 encoding of the repetition code.

**Checkpoint:**  Suppose you sent the logical code word 0110110, but the recipient received the message 0110100.  We'll assume that at most only one error occurred.  Would the recipient be able to identify if there was an error?  If so, could they locate the error?  
Hint: errors could occur on any of the data or the parity bits.

In practice, a large message can be broken into blocks with each block is encoded using the Hamming code.  The code scales much better than the repetition code.  A Hamming code can be produced for any integer $r$ greater than 1, such that the code is characterized as $[2^r-1,2^r -r -1,3]$.  So for a message of size $2^5 -5 -1 = 26$, the Hamming code would require only $2^5-1 =31$ bits while a three bit repetition code would require $26*3 = 78$ bits.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2:</span>** The Matrix Form of the Hamming Code

The Hamming code is commonly constructed with special matrices so a few simple linear algebra operations can encode and decode messages. The next two cells will have you define these matrices and see if you can reproduce the example above.

</div>



First, define the generator matrix $G$ such that a dot product between the message and $G$  mod2 performs the valid encoding. Hint: G should be a 4x7 matrix. 

In [ ]:
# EXERCISE 2

message = np.array([0, 1, 1, 0])

# The G matrix should properly encode the message when the following calculation is performed
message = np.array([0, 1, 1, 0])

G = np.array([
    [1, 0, 0, 0, 1, 1, 0],
    [0, 1, 0, 0, 1, 0, 1],
    [0, 0, 1, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1]
])

encoded = np.dot(message, G) % 2

print(encoded)


Now, define the parity check matrix $H$ such that $Hv \mod 2$ produces a syndrome, where $v$ is the received message vector. 

In [ ]:
received = np.array([0, 0, 1, 0,1,1,0]) # error applied on second data qubit
print(received)


# Define the parity check matrix H
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1]
])

decoded = np.dot(H,received) % 2
print(decoded)


---

## 1.4 What Makes QEC so Hard?

QEC shares the same goal as classical EC, but comes with a number of unique challenges thanks to the properties of quantum mechanics.  This section will list the primary differences, and the following section will explain how these challenges can be addressed.
 
1.  Continuous Errors - Classical errors are always discrete bit flips. Quantum errors are continuous and can manifest in an infinite number of ways, potentially shifting a qubit's state to any point on the Bloch sphere. For instance, the figure below illustrates many possible errors that affect a qubit starting in the $\ket{0}$ state. Errors can perturb states incoherently (from environmental effects) or coherently from slight hardware imperfections. This invites the question, "Does QEC require an infinite amount of resources to correct errors?"
   
<img src="../Images/c_error.png" alt="Bloch sphere showing a qubit initially in the zero state with multiple arrows pointing to various locations on the sphere, illustrating how continuous quantum errors can shift a qubit state to any point" style="width: 300px;"/>

2. No Cloning - Quantum states cannot be copied. That is to say that the following expression holds:$~\nexists U \text{ such that } U(\ket{\psi} \otimes \ket{\rho}) =  \ket{\psi} \otimes\ket{\psi}$. This means we cannot just send multiple copies of the quantum state through the noisy channel like the classical repetition code. 

3. Destructive Measurement - In classical EC, the state can be accessed at any time, making decoding much easier. Measuring a quantum state collapses it, making the EC moot if the state is destroyed. Therefore, more clever ways to extract syndromes are required. A secondary consequence of this fact is sampling error.  Even if an algorithm could perform perfectly ensuring no sources of error, many applications require statistical sampling of the resulting state. If we sampled $\ket{\psi} = \alpha\ket{0} + \beta\ket{1}$ the frequency of 0's would be close to $\alpha^2$ but deviate based on the number of samples per the Central Limit Theorem.

4. Scalability - Though scalability is an issue for classical EC, it is far more severe for QEC. Today's noisy intermediate scale quantum devices are very difficult to control, so each additional qubit required for QEC comes at great cost.  Qubits also have short coherence times, so QEC procedures must complete within strict time constraints which gets harder at scale. Finally, the threshold theorem is in play. In classical EC, adding more bits always reduces the logical error rate. This is not true for quantum - physical qubits must have noise below a specific threshold in order for scaling the code to improve the error rates,  otherwise, the results just get worse.


---

## 1.5 There is Still Hope for QEC!

The challenges discussed above are daunting but there are many ingenious techniques developed to help circumvent them. That said, practical QEC remains difficult to realize and is an extremely active research field - viewed as one of the most important prerequisites for useful quantum computing.  This section will begin to bridge the gap between classical EC and QEC.

### Syndrome Extraction

The no cloning principle means quantum states cannot be copied for QEC. We'll need a clever way to extract syndromes from the logical state that does not rely on repetition. But, how is this done without destroying the information that is being protected?

The solution involves **stabilizers** which are specially designed operators that act on a logical state without changing it, but still enable us to learn about errors by performing projective measurement of ancilla qubits.  The next notebook in this series will introduce stabilizers with more mathematical rigor, and the following section of this lab will provide a more concrete example of a simple stabilizer in action.  Essentially, stabilizers perform parity checks and project the quantum state into the $1\ket{\psi}$ state if the parity check passes and $-1\ket{\psi}$ if the parity check is violated. So, you return the same state either way, and with enough atabilizers, can identify which errors occured and fix them.


### Digitization of errors

Errors can perturb states incoherently from environmental effects or coherently from slight hardware imperfections.  While both types of errors can be addressed, we’ll focus on coherent errors first because they’re often easier to isolate and analyze.

For instance, a rotation gate that should be at an angle of $\frac{\pi}{16} \approx 0.196 $ ends up being more like 0.17. This may seem inconsequential, but imperfections like this accumulate and quickly ruin the outcome of a quantum algorithm.



We will focus on three different coherent errors that can occur on a qubit storing the quantum state $\ket{\psi} = \alpha \ket{0}+\beta\ket{1}$:

* **Bit flip errors (X)** swap a qubit's amplitudes, transforming $\ket{\psi}$ to $\beta\ket{0}+\alpha\ket{1}$.
* **Phase flip errors (Z)** introduce a sign change in one of the amplitudes, transforming $\ket{\psi}$ to $\alpha\ket{0}-\beta\ket{1}$.
* **Combining a bit flip with a phase flip error (X and Z)** swaps amplitudes and applies a sign change, transforming $\ket{\psi}$ to $\beta\ket{0}-\alpha\ket{1}$.

Once we have identified one of these errors, we can correct it. For instance, if a qubit has undergone a bit flip error, we can correct it by applying an $X$ gate. Similarly, to correct a qubit that has experienced a phase flip error, we simply apply a $Z$ gate. How would you correct a qubit that has been identified as having undergone a bit flip error followed by a phase flip error?  

We can address all coherent errors with a key insight: although the Bloch sphere suggests errors can occur through infinitely many possible rotations, all such errors can be broken down into three basic forms &mdash; bit flips, phase flips, or a combination of both bit flips and phase flips.  In other words, any error state can be reached by some combination of $X$, $Y$, and $Z$ rotations. So, why are there not an infinite number of corrections?

The math below explains how an arbitrary error results in a finite set of possible error states. 

 Consider a qubit in the following normalized state.
 $$ \ket{\psi}  =  \cos\frac{\theta}{2}\ket{0} + e^{i\phi}\sin\frac{\theta}{2}\ket{1} $$
 
  Coherent errors can be represented by the application of a Unitary $U(\delta\theta,\delta\phi)$ which acts on the ideal state and perturbs it.
$$ U(\delta\theta,\delta\phi)\ket{\psi}  =  \cos\frac{\theta +\delta\theta}{2}\ket{0} + e^{i\phi+\delta\phi}\sin\frac{\theta+\delta\theta}{2}\ket{1} $$
 Using the fact that the Pauli matrices form a basis for any 2x2 unitary matrix and taking advantage of the identity $Y=iXZ$, the operation can be rewritten as
 $$ U(\delta\theta,\delta\phi) \ket{\psi} = \alpha_II\ket{\psi} +\alpha_X X\ket{\psi}+\alpha_Z Z\ket{\psi}+\alpha_{XZ}XZ\ket{\psi}   $$
 This means that any coherent error can be **digitized** into X-type bit flip errors ($X \ket{\psi} = \alpha X\ket{0} + \beta X\ket{1} = \alpha\ket{1} + \beta\ket{0}$), Z-type phase flip errors ($Z\ket{\psi} = \alpha Z\ket{0} + \beta Z\ket{1} = \alpha\ket{0} - \beta\ket{1}$), or a combination of the two (XZ). This makes correction much more tractable, as there are only three types of errors to consider.

 Try the widget [linked here](https://nvidia.github.io/cuda-q-academic/interactive_widgets/error_digitization.html) to see a concrete example of this for an $X$ rotation (bitflip error) impacting the state $\ket{000}$ and how stabilizers are they key to correcting the error.

### Better QEC Codes and AI Solutions

Finally, overcoming the QEC scaling challenges will require breakthroughs on many fronts. Significant research efforts are targeting discovery of more efficient QEC codes that require fewer qubits. AI is already showing great promise as a tool to help find new QEC codes, and accelerate decoding. Later notebooks will explore AI for QEC applications.

---

## 1.6 The Quantum Repetition Code

A quantum state cannot be cloned, but it can be redundantly encoded across additional entangled qubits. Let's start with a generic normalized qubit state $\ket{\psi}$:

$$\ket{\psi} = \alpha\ket{0} +\beta\ket{1}.$$ 

The 0 and 1 states can be encoded into a logical state making use of the larger 8-dimensional Hilbert space of three qubits:  

$$\ket{\psi}_L = \alpha\ket{000} +\beta\ket{111} = \alpha\ket{0}_L +\beta\ket{1}_L.$$ 

Note that this is *not* equivalent to $\ket{\psi} \otimes \ket{\psi} \otimes \ket{\psi}$.

Consider now the entire Hilbert space spanned by the eight basis states.  The basis states are separated into the logical codewords that make up the codespace:   

| Codespace    | Notation for the logical codewords| 
| ----------- | ----------- | 
| $\ket{000}$ | $\ket{0}_L$ |
|$\ket{111}$ | $\ket{1}_L$ |

and the remaining basis states make up the error space:

| Error space | 
| ----------- | 
| $\ket{001}$ | 
| $\ket{010}$ |
| $\ket{100}$ |
| $\ket{011}$ |
| $\ket{101}$ |
| $\ket{110}$ |

Assume that the state $\ket{111}$ is transmitted through a noisy channel and becomes $\ket{011}$?  How might it be decoded to tell which logical codeword it is closest to?  Remember, you cannot simply examine the state and perform a majority count as no information about the state is accessible without some sort of measurement that induces wavefunction collapse.  

Consider the operators $Z_1Z_2$ and $Z_2Z_3$.  (Remember that $Z_n$ returns an eigenvalue of +1 if the nth qubit in a ket is a 0 and -1 if it is a 1).  
It turns out that these operators have special properties such that the eigenvalues produced when they act on any of the states in the codespace or error space can be extracted with ancilla qubits without disturbing the state.  This means, there is a way to extract parity check information just like the classical Hamming code!

Note: Operators with these "special properties" are called stabilizers and will be rigorously introduced in the next lab.

The details of this extraction process are described shortly, but the implication is that syndromes can be produced from the parity check results and used to identify corrections to the quantum repetition code without destroying the encoded state. Considering only single bit flip errors, the table below shows the possible syndromes, the corresponding errors, and the operation that can be applied to correct the error assuming $\ket{111}$ is the transmitted message.  

| $Z_1Z_2$ Syndrome   | $Z_2Z_3$ Syndrome| Encoded State | Single Bit Flip Correction 
| ----------- | ----------- | ----------- | ----------- | 
| 0 | 0 | $\ket{111}$ | none |
| 1 | 0 | $\ket{011}$ | $X_1$ |
| 0 | 1 | $\ket{110}$ | $X_3$ |
| 1 | 1 | $\ket{101}$ | $X_2$ |


Like the classical repetition code, there is no way to know for certain if a 10 syndrome corresponds to a single bit flip error from the $\ket{111}$ codeword or a two bit flip error from the $\ket{000}$ codeword.  However, it is always prudent to assume that the case with fewer errors is more likely.

The entire quantum circuit for the three qubit repetition code is shown below, where the ancilla qubits are used to compute the $Z_1Z_2$ and $Z_2Z_3$ syndromes.



<img src="../Images/repcircuit.png" alt="Quantum circuit diagram for the three-qubit repetition code showing three data qubits, custom identity gates for noise injection, Hadamard-controlled-Z-Hadamard syndrome extraction circuits on two ancilla qubits, and conditional X corrections based on measured syndromes" style="width: 800px;"/>



It is helpful to explore in greater detail how the syndromes can be extracted using the ancilla qubits without disturbing the state.
First, consider the initial state of the first ancilla after application of a Hadamard gate and the encoded state after a bit flip has occurred on the first data qubit.

$$ \frac{1}{\sqrt{2}}(\ket{0} + \ket{1})\ket{011} $$

Next, a controlled $Z_1Z_2$ operation is applied to the data qubits resulting in

$$ \frac{1}{\sqrt{2}}(\ket{0}\ket{011} + \ket{1}Z_1Z_2\ket{011}), $$

followed by an application of the second Hadamard:

$$ \frac{1}{2} ((\ket{0}+\ket{1})\ket{011} + (\ket{0} -\ket{1})Z_1Z_2\ket{011})  =  \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  .   $$

Now, if $Z_1Z_2\ket{011}$ is evaluated, the result is $-\ket{011}$ and the entire state simplifies to $\ket{1}\ket{011}$ meaning upon measurement, the first ancilla qubit will be measured as 1 with certainty and the data qubits remain undisturbed in the $\ket{011}$ state:


$$ \ket{0}(\frac{1+Z_1Z_2}{2})\ket{011} + \ket{1}(\frac{1-Z_1Z_2}{2})\ket{011}  =   \ket{0}(\frac{1+ -1}{2})\ket{011} + \ket{1}(\frac{1--1}{2})\ket{011}  =   \ket{1}\ket{011}.  $$

A similar analysis will show that the second ancilla qubit will be measured as 0 with certainty without distubring the data qubits.  Accordoing to the syndrome table, 
this will trigger an application of the $X$ gate on the first qubit to correct the error.


---

## 1.7 Coding the Quantum Repetition Code

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3:</span>** The Quantum Repetition Code

Now that you understand the quantum repetition code, try to code it using CUDA-Q. Fill in each of the steps below marked `##TODO##`. CUDA-Q contains a couple of features particularly helpful for building QEC workflows. First, and already completed for you, is the definition of a custom noise model which produces custom identity operations that can randomly perform bit flips on specific qubits. Second, you can measure the ancilla qubits within the kernel and use the result to perform a correction operation. The documentation example on [building kernels](https://nvidia.github.io/cuda-quantum/latest/using/examples/building_kernels.html) and [mid-circuit measurement](https://nvidia.github.io/cuda-quantum/latest/examples/python/measuring_kernels.html) may be helpful for this exercise.

Try to code all the steps and then sample the kernel to determine the logical error rate.

</div>

In [ ]:
# EXERCISE 3

cudaq.set_target('density-matrix-cpu')


#First, create an empty noise model
noise_model = cudaq.NoiseModel()
p = 0.2

#Build a custom gate which applies the identity operation
cudaq.register_operation("custom_i", np.array([1, 0, 0, 1]))

#Add a bitflip noise channel to the custom_i gate applied to each qubit
noise_model.add_channel("custom_i", [0], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [1], cudaq.BitFlipChannel(p))
noise_model.add_channel("custom_i", [2], cudaq.BitFlipChannel(p))

@cudaq.kernel
def three_qubit_repetition_code() -> list[int]:
    data_qubits = cudaq.qvector(3)
    ancilla_qubits = cudaq.qvector(2)

    # Initialize the logical |1> state as |111>
    x(data_qubits)

    # Random Bit Flip Errors
    for i in range(3):
        custom_i(data_qubits[i])

    # Extract Syndromes

    h(ancilla_qubits)

    # First Parity Check
    z.ctrl(data_qubits[0],ancilla_qubits[0])
    z.ctrl(data_qubits[1],ancilla_qubits[0])

    # Second Parity Check
    z.ctrl(data_qubits[1],ancilla_qubits[1])
    z.ctrl(data_qubits[2],ancilla_qubits[1])

    h(ancilla_qubits)

    s0 = mz(ancilla_qubits[0])
    s1 = mz(ancilla_qubits[1])


    # Correct errors based on syndromes
    if s0 and s1:
        x(data_qubits[1])
    elif s0:
        x(data_qubits[0])
    elif s1:
        x(data_qubits[2])

    d0 = mz(data_qubits[0])
    d1 = mz(data_qubits[1])
    d2 = mz(data_qubits[2])

    return [d0, d1, d2, s0, s1]

# Run the kernel and observe results
# The percent of samples that are 000 on first three qubits (data qubits) corresponds to the logical error rate
result = cudaq.run(three_qubit_repetition_code, noise_model=noise_model, shots_count =100)

logical_errors = 0
for run in result:
    if run[0] or run[1] or run[2]:
        logical_errors += 1
    print(run)

print("Logical Error Rate:",1 - logical_errors/100)

## Conclusion

You now have a basic understanding of error correction (EC) and quantum error correction (QEC). You explored the five aspects common to all EC procedures, implemented the classical repetition code and Hamming code, identified the unique challenges of QEC, and built the three-qubit quantum repetition code in CUDA-Q. The next lab will explore **stabilizers** in more detail and equip you to code two of the most famous and fundamental QEC codes: the Shor code and the Steane code. Future labs will cover more advanced topics like decoding and other specific QEC codes.

**Related Notebooks:**
* [QEC 101 Lab 2 — Stabilizers](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb) — introduces the stabilizer formalism for quantum error correction
* [Quick Start to Quantum — Notebook 1](https://github.com/NVIDIA/cuda-q-academic/blob/main/quick-start-to-quantum/01_quick_start_to_quantum.ipynb) — prerequisite notebook covering qubits, gates, and measurement in CUDA-Q
* [QEC 101 Lab 3 — Noisy Simulation](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/03_QEC_Noisy_Simulation.ipynb) — applies QEC codes with realistic noise models in CUDA-Q